# Sharpe & Sortino — companion notebook

Companion for [`sharpe-sortino.md`](./sharpe-sortino.md). Same Sharpe, very different Sortino — demonstrating that volatility hides asymmetry.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(0)

In [ ]:
def sharpe(r, rf=0, ann=252):
    e = r - rf/ann
    return np.mean(e)/np.std(e, ddof=1) * np.sqrt(ann)
def sortino(r, tau=0, ann=252):
    e = r - tau/ann
    downside = np.sqrt(np.mean(np.minimum(e, 0)**2))
    return np.mean(e)/downside * np.sqrt(ann)

T = 2520  # 10 years of daily
# 1) Symmetric: Normal
r_norm = 0.0003 + 0.012*rng.standard_normal(T)
# 2) Same mean & vol, but with occasional big losses (negative skew)
r_skew = 0.0008 + 0.012*rng.standard_normal(T)
ix = rng.choice(T, size=T//40, replace=False)
r_skew[ix] -= 0.04  # rare large drawdowns

for name, r in [('Normal',r_norm), ('Negatively skewed (same vol)', r_skew)]:
    print(f'{name:<32s}  Sharpe={sharpe(r):.2f}  Sortino={sortino(r):.2f}  '
          f'mean={r.mean()*252:.2%}  vol={r.std(ddof=1)*np.sqrt(252):.2%}  '
          f'skew={stats.skew(r):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, r, name in zip(axes, [r_norm, r_skew], ['Normal', 'Negatively skewed']):
    ax.hist(r, bins=80, color='tab:blue', alpha=0.7)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f'{name} (Sharpe={sharpe(r):.2f}, Sortino={sortino(r):.2f})')
    ax.set_xlabel('daily return')
plt.tight_layout(); plt.show()